# Example 04: Multi-Agent Orchestration

**Goal:** Build Polly-style orchestrators that delegate work to sub-agents running in parallel.

**Pattern:** Orchestrator plans and delegates, sub-agents implement, cross-vendor review.

## 1. The Orchestrator Pattern

In [ ]:
orchestrator_diagram = '''
ORCHESTRATOR (claude-sdk)
│
├── sys_session_send → SUB-AGENT A (implements feature X)
│     └── Opens PR in its own worktree
│
├── sys_session_send → SUB-AGENT B (implements feature Y)
│     └── Opens PR in its own worktree
│
├── WAIT (inbox-driven — no polling!)
│
├── sys_read_inbox → Both done
│
├── sys_session_send → REVIEWER (different vendor, reviews both PRs)
│
└── PRESENT RESULTS → Human merges
'''
print(orchestrator_diagram)

## 2. Sub-Agent Definitions

Define sub-agents as tools in the orchestrator's YAML:

In [ ]:
sub_agent_def = '''
tools:
  # Sub-agent 1: Coder (Claude)
  coder:
    type: agent
    harness: claude-sdk
    model: claude-sonnet-4-6
    prompt: |
      You are a coding specialist. Write clean, tested code.
      Always write tests first. Open a PR when done.
    os_env:
      type: caller_process
      cwd: .
      sandbox:
        type: none

  # Sub-agent 2: Reviewer (GPT)
  reviewer:
    type: agent
    harness: openai-agents
    model: gpt-5
    prompt: |
      You review code for correctness, style, and security.
      Never make edits — report issues only.
    os_env:
      type: caller_process
      cwd: .
      sandbox:
        type: none
'''
print("Sub-agent definitions (go inside tools: in orchestrator YAML):")
print(sub_agent_def)

## 3. Minimal Orchestrator YAML

In [ ]:
orchestrator_yaml = '''
name: mini_orchestrator
description: Simple orchestrator that delegates coding and review.

prompt: |
  You are an orchestrator. You NEVER write code yourself —
  ALL coding work is delegated to your `coder` sub-agent.

  For every task:
  1. Plan what needs to be built
  2. Dispatch to `coder` via sys_session_send
  3. Collect result via sys_read_inbox (don't poll!)
  4. Dispatch to `reviewer` for review
  5. Present both to the user

  Important: The reviewer must be a DIFFERENT vendor than the coder.

executor:
  harness: claude-sdk
  model: claude-sonnet-4-6

os_env:
  type: caller_process
  cwd: .
  sandbox:
    type: none

async: true
cancellable: true
spawn: true

tools:
  coder:
    type: agent
    harness: claude-sdk
    model: claude-sonnet-4-6
    prompt: |
      You are a coding specialist. Write clean, tested code.
      Open a PR for every change.
    os_env:
      type: caller_process
      cwd: .
      sandbox: {type: none}

  reviewer:
    type: agent
    harness: openai-agents
    model: gpt-5
    prompt: |
      You review code. Check: correctness, style, tests, security.
      Report issues; never edit.
    os_env:
      type: caller_process
      cwd: .
      sandbox: {type: none}
'''

with open('agent_orchestrator.yaml', 'w') as f:
    f.write(orchestrator_yaml)
print("Created agent_orchestrator.yaml")
print("Run: omni run agent_orchestrator.yaml")

## 4. Polly: The Full Orchestrator (Reference)

Polly is Omnigent's shipped orchestrator. Key design decisions:

| Feature | Polly's implementation |
|---|---|
| **Roster preflight** | Checks which worker CLIs are on PATH before dispatching |
| **Parallel worktrees** | Each sub-agent gets a `git worktree` — isolated but same repo |
| **Cross-vendor review** | Reviewer ALWAYS different vendor than implementer |
| **Inbox-driven** | Orchestrator sleeps between dispatches, woken by sub-agent completion |
| **PR per task** | Each implementer opens its own PR; human merges |
| **Blast radius** | Guardrails prevent force-push, rm -rf /, hard-reset |
| **Purpose guard** | Sub-agents restricted to: implement, review, explore, search |

**Next:** [Example 05 — Sandboxing & Security](./example_05_sandbox.ipynb)